# AstroCLIMB 01 — data audit and metadata retrieval

This notebook is the first stage of the system. It does **not** train a classifier and it never uses test labels. It:

1. locates and validates the Kaggle competition files;
2. builds a fresh compact index from the official `adsabs/AstroCLIMB` metadata;
3. retrieves source records using normalized captions and targeted image hashes;
4. resolves relationships from figure identity, DOI equality, and citation edges;
5. measures train accuracy/coverage and test coverage; and
6. saves compact audit CSVs for later notebooks.

## Before running on Kaggle

- Attach the AstroCLIMB competition data.
- Use a GPU only if convenient; this notebook is CPU/network/I/O bound.
- Turn **Internet on** so Hugging Face streaming works, or attach a local Parquet copy and set `METADATA_PATH`.
- A complete targeted image pass streams the full Hugging Face image column (roughly 72 GB), so the first run can take hours. Save `/kaggle/working/astroclimb_01` as a private Kaggle Dataset afterward.
- Leave `ENABLE_PIXEL_HASH=False` initially. Enable it only if raw image matching is inadequate.

The local `solution.csv` is deliberately ignored.


In [1]:
%pip install -q "datasets>=3.0" pyarrow rapidfuzz


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import base64, gc, hashlib, io, json, os, pickle, re, struct, unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from rapidfuzz.fuzz import ratio as fuzzy_ratio

LABELS = ["same_figure", "same_paper", "related_papers", "unrelated_papers"]
HF_DATASET = "adsabs/AstroCLIMB"
HF_SPLIT = "train"
METADATA_PATH = None  # Example: Path('/kaggle/input/astroclimb-hf/parquet')
WORK = Path('/kaggle/working/astroclimb_01')
WORK.mkdir(parents=True, exist_ok=True)
COMPETITION_DIR = Path('/kaggle/input/competitions/astroclimb')
TRAIN_PATH = COMPETITION_DIR / 'train.csv'
TEST_PATH = COMPETITION_DIR / 'test.csv'
SAMPLE_PATH = COMPETITION_DIR / 'sample_submission.csv'

CSV_CHUNK = 32
IMAGE_STREAM_BATCH = 16
REBUILD_METADATA_INDEX = True
RUN_IMAGE_MATCHING = True
ENABLE_PIXEL_HASH = False
ENABLE_FUZZY_CAPTIONS = True
FUZZY_MIN_SCORE = 98.5
FUZZY_MIN_MARGIN = 2.0
MAX_FUZZY_CANDIDATES = 1500
print('Output directory:', WORK)


Output directory: /kaggle/working/astroclimb_01


## Locate and validate competition files

Only exact `train.csv`, `test.csv`, and `sample_submission.csv` filenames are accepted. The largest candidate is selected, which avoids accidentally using a 1,000-row smoke-test slice.


In [3]:
def find_competition_file(name):
    hits = []
    for root in [Path('/kaggle/input'), Path('data'), Path('.')]:
        if root.exists():
            hits.extend(p for p in root.rglob(name) if p.is_file())
    hits = sorted(set(hits), key=lambda p: (-p.stat().st_size, len(p.parts), str(p)))
    if not hits:
        raise FileNotFoundError(f'Could not find {name}. Attach the competition data.')
    print(name, '->', hits[0], f'({hits[0].stat().st_size / 2**30:.2f} GiB)')
    if len(hits) > 1:
        print('  other candidates:', [str(p) for p in hits[1:4]])
    return hits[0]

TRAIN_CSV = TRAIN_PATH if TRAIN_PATH.is_file() else find_competition_file('train.csv')
TEST_CSV = TEST_PATH if TEST_PATH.is_file() else find_competition_file('test.csv')
SAMPLE_CSV = SAMPLE_PATH if SAMPLE_PATH.is_file() else find_competition_file('sample_submission.csv')
print('train ->', TRAIN_CSV)
print('test ->', TEST_CSV)
print('sample submission ->', SAMPLE_CSV)

def header(path):
    return pd.read_csv(path, nrows=0).columns.tolist()

assert {'id', 'obj_1', 'obj_2', *LABELS}.issubset(header(TRAIN_CSV))
assert {'id', 'obj_1', 'obj_2'}.issubset(header(TEST_CSV))
assert {'id', *LABELS}.issubset(header(SAMPLE_CSV))
assert not set(LABELS).intersection(header(TEST_CSV)), 'Test labels must not be present'
print('File schemas are valid.')


train -> /kaggle/input/competitions/astroclimb/train.csv
test -> /kaggle/input/competitions/astroclimb/test.csv
sample submission -> /kaggle/input/competitions/astroclimb/sample_submission.csv
File schemas are valid.


## Streaming data audit

This reads each large CSV once without retaining Base64 images in memory. It also collects the unique image hashes needed for targeted metadata matching.


In [4]:
def is_image(value):
    return isinstance(value, str) and value.lstrip().startswith(('iVBOR', 'data:image/'))

def decode_image_bytes(value):
    payload = value.strip()
    if payload.startswith('data:image/'):
        payload = payload.split(',', 1)[1]
    return base64.b64decode(payload, validate=False)

def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()

def pixel_hash(raw):
    with Image.open(io.BytesIO(raw)) as image:
        image = ImageOps.exif_transpose(image).convert('RGB')
        size = struct.pack('>II', image.width, image.height)
        return sha256_bytes(size + image.tobytes())

def audit_csv(path, labeled):
    counts, modalities, raw_hashes, pixel_hashes = Counter(), Counter(), set(), set()
    rows = invalid_images = 0
    cols = ['id', 'obj_1', 'obj_2'] + (LABELS if labeled else [])
    for chunk_no, chunk in enumerate(pd.read_csv(path, usecols=cols, chunksize=CSV_CHUNK, keep_default_na=False)):
        rows += len(chunk)
        if labeled:
            active = chunk[LABELS].sum(axis=1)
            if not active.eq(1).all():
                raise ValueError('Training rows must have exactly one active label')
            counts.update(chunk[LABELS].idxmax(axis=1))
        for a, b in zip(chunk.obj_1, chunk.obj_2):
            ai, bi = is_image(a), is_image(b)
            modalities['image-image' if ai and bi else 'text-image' if ai ^ bi else 'text-text'] += 1
            for value, flag in ((a, ai), (b, bi)):
                if not flag:
                    continue
                try:
                    raw = decode_image_bytes(value)
                    raw_hashes.add(sha256_bytes(raw))
                    if ENABLE_PIXEL_HASH:
                        pixel_hashes.add(pixel_hash(raw))
                except Exception:
                    invalid_images += 1
        if chunk_no % 100 == 0:
            print(path.name, 'rows audited:', rows)
    return {
        'rows': rows, 'class_counts': dict(counts), 'modality_counts': dict(modalities),
        'invalid_image_occurrences': invalid_images, 'raw_hashes': raw_hashes,
        'pixel_hashes': pixel_hashes,
    }

train_audit = audit_csv(TRAIN_CSV, True)
test_audit = audit_csv(TEST_CSV, False)
sample_rows = sum(len(c) for c in pd.read_csv(SAMPLE_CSV, usecols=['id'], chunksize=5000))
assert test_audit['rows'] == sample_rows
print('Train:', {k:v for k,v in train_audit.items() if not k.endswith('hashes')})
print('Test:', {k:v for k,v in test_audit.items() if not k.endswith('hashes')})
TARGET_RAW_HASHES = train_audit['raw_hashes'] | test_audit['raw_hashes']
TARGET_PIXEL_HASHES = train_audit['pixel_hashes'] | test_audit['pixel_hashes']
print('Unique target images:', len(TARGET_RAW_HASHES))


train.csv rows audited: 32
train.csv rows audited: 3232
train.csv rows audited: 6432
train.csv rows audited: 9632
test.csv rows audited: 32
test.csv rows audited: 3232
test.csv rows audited: 6432
test.csv rows audited: 9632
Train: {'rows': 10000, 'class_counts': {'same_figure': 1000, 'same_paper': 3000, 'related_papers': 3000, 'unrelated_papers': 3000}, 'modality_counts': {'text-image': 4000, 'image-image': 3000, 'text-text': 3000}, 'invalid_image_occurrences': 0}
Test: {'rows': 10000, 'class_counts': {}, 'modality_counts': {'text-image': 4000, 'image-image': 3000, 'text-text': 3000}, 'invalid_image_occurrences': 0}
Unique target images: 14598


## Build a fresh compact metadata index

The first pass selects metadata columns only, so it does not transfer the large image column. Captions are retained in normalized form for conservative fuzzy recovery.


In [5]:
INDEX_PATH = WORK / 'metadata_index.pkl'

def normalize_caption(value):
    text = unicodedata.normalize('NFKC', str(value or ''))
    return re.sub(r'\s+', ' ', text).strip().casefold()

def relaxed_caption(value):
    text = normalize_caption(value)
    text = text.replace('﹩', '$').replace('−', '-').replace('–', '-')
    text = re.sub(r'\\(?:mathrm|text|rm)\s*\{([^{}]*)\}', r'\1', text)
    text = re.sub(r'[^\wα-ωΑ-Ω]+', ' ', text, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', text).strip()

def normalize_doi(value):
    doi = str(value or '').strip().casefold()
    doi = re.sub(r'^https?://(?:dx\.)?doi\.org/', '', doi)
    return doi.rstrip('.,; ')

def doi_set(value):
    if value is None:
        return frozenset()
    if isinstance(value, str):
        try:
            value = json.loads(value) if value.strip().startswith('[') else [value]
        except json.JSONDecodeError:
            value = [value]
    try:
        values = list(value)
    except TypeError:
        values = [value]
    return frozenset(filter(None, (normalize_doi(x) for x in values)))

def iter_metadata(columns, batch_size=256):
    if METADATA_PATH is not None:
        import pyarrow.dataset as pads
        dataset = pads.dataset(str(METADATA_PATH), format='parquet')
        missing = set(columns) - set(dataset.schema.names)
        if missing:
            raise ValueError(f'Metadata is missing columns: {sorted(missing)}')
        for batch in dataset.scanner(columns=columns, batch_size=batch_size, use_threads=False).to_batches():
            for row in batch.to_pylist():
                yield row
    else:
        from datasets import load_dataset
        stream = load_dataset(HF_DATASET, split=HF_SPLIT, streaming=True)
        missing = set(columns) - set(stream.column_names)
        if missing:
            raise ValueError(f'Hugging Face metadata is missing columns: {sorted(missing)}')
        yield from stream.select_columns(columns)

def append_unique(mapping, key, row_id):
    if key and row_id not in mapping[key]:
        mapping[key].append(row_id)

if INDEX_PATH.exists() and not REBUILD_METADATA_INDEX:
    with INDEX_PATH.open('rb') as f:
        index = pickle.load(f)
    print('Loaded cached index:', INDEX_PATH)
else:
    records, caption_exact, caption_relaxed = [], defaultdict(list), defaultdict(list)
    columns = ['UUID', 'Image ID', 'Paper DOI', 'Image Caption', 'References DOIs', 'Citing DOIs']
    for row in iter_metadata(columns):
        caption = normalize_caption(row.get('Image Caption'))
        rec = {
            'meta_row': len(records), 'uuid': str(row.get('UUID') or ''),
            'image_id': str(row.get('Image ID') or ''),
            'doi': normalize_doi(row.get('Paper DOI')),
            'caption': caption, 'relaxed_caption': relaxed_caption(caption),
            'references': doi_set(row.get('References DOIs')),
            'citations': doi_set(row.get('Citing DOIs')),
        }
        records.append(rec)
        append_unique(caption_exact, rec['caption'], rec['meta_row'])
        append_unique(caption_relaxed, rec['relaxed_caption'], rec['meta_row'])
        if len(records) % 10000 == 0:
            print('Metadata rows indexed:', len(records))
    index = {
        'records': records, 'caption_exact': dict(caption_exact),
        'caption_relaxed': dict(caption_relaxed), 'raw_images': {}, 'pixel_images': {},
        'hf_dataset': HF_DATASET, 'hf_split': HF_SPLIT,
    }
    with INDEX_PATH.open('wb') as f:
        pickle.dump(index, f, protocol=pickle.HIGHEST_PROTOCOL)
    print('Saved fresh metadata index:', INDEX_PATH)
print({k: len(index[k]) for k in ['records', 'caption_exact', 'caption_relaxed', 'raw_images', 'pixel_images']})


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/114 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/114 [00:00<?, ?it/s]

Metadata rows indexed: 10000
Metadata rows indexed: 20000
Metadata rows indexed: 30000
Metadata rows indexed: 40000
Metadata rows indexed: 50000
Metadata rows indexed: 60000
Metadata rows indexed: 70000
Metadata rows indexed: 80000
Metadata rows indexed: 90000
Saved fresh metadata index: /kaggle/working/astroclimb_01/metadata_index.pkl
{'records': 94233, 'caption_exact': 92801, 'caption_relaxed': 92776, 'raw_images': 0, 'pixel_images': 0}


## Targeted image matching

Only hashes that occur in train/test are retained, but the entire metadata image split is scanned so duplicate images remain correctly marked as ambiguous. This cell is the slow part.


In [6]:
def metadata_image_bytes(value, base_dir=Path('.')):
    if value is None:
        return None
    if isinstance(value, (bytes, bytearray, memoryview)):
        return bytes(value)
    if isinstance(value, dict):
        if value.get('bytes') is not None:
            return bytes(value['bytes'])
        if value.get('path'):
            path = Path(value['path'])
            return (path if path.is_absolute() else base_dir / path).read_bytes()
    if isinstance(value, Image.Image):
        buf = io.BytesIO(); value.save(buf, format='PNG'); return buf.getvalue()
    return None

def iter_metadata_images():
    if METADATA_PATH is not None:
        import pyarrow.dataset as pads
        dataset = pads.dataset(str(METADATA_PATH), format='parquet')
        columns = ['image'] + (['UUID'] if 'UUID' in dataset.schema.names else [])
        base_dir = Path(METADATA_PATH) if Path(METADATA_PATH).is_dir() else Path(METADATA_PATH).parent
        for batch in dataset.scanner(columns=columns, batch_size=IMAGE_STREAM_BATCH, use_threads=False).to_batches():
            for row in batch.to_pylist():
                yield row, base_dir
    else:
        from datasets import Image as HFImage, load_dataset
        stream = load_dataset(HF_DATASET, split=HF_SPLIT, streaming=True)
        stream = stream.cast_column('image', HFImage(decode=False))
        columns = ['image'] + (['UUID'] if 'UUID' in stream.column_names else [])
        for row in stream.select_columns(columns):
            yield row, Path('.')

if RUN_IMAGE_MATCHING:
    uuid_to_row = {r['uuid']: i for i, r in enumerate(index['records']) if r['uuid']}
    raw_map = defaultdict(list, {k:list(v) for k,v in index.get('raw_images', {}).items()})
    pixel_map = defaultdict(list, {k:list(v) for k,v in index.get('pixel_images', {}).items()})
    raw_found = pixel_found = scanned = 0
    for sequential_row, (row, base_dir) in enumerate(iter_metadata_images()):
        scanned += 1
        raw = metadata_image_bytes(row.get('image'), base_dir)
        if raw:
            rh = sha256_bytes(raw)
            if rh in TARGET_RAW_HASHES:
                before = len(raw_map[rh])
                append_unique(raw_map, rh, uuid_to_row.get(str(row.get('UUID') or ''), sequential_row))
                raw_found += len(raw_map[rh]) > before
            if ENABLE_PIXEL_HASH:
                try:
                    ph = pixel_hash(raw)
                    if ph in TARGET_PIXEL_HASHES:
                        before = len(pixel_map[ph])
                        append_unique(pixel_map, ph, uuid_to_row.get(str(row.get('UUID') or ''), sequential_row))
                        pixel_found += len(pixel_map[ph]) > before
                except Exception:
                    pass
        if scanned % 1000 == 0:
            print('Images scanned:', scanned, 'raw matches:', raw_found, 'pixel matches:', pixel_found)
    index['raw_images'], index['pixel_images'] = dict(raw_map), dict(pixel_map)
    index['image_scan_complete'] = True
    with INDEX_PATH.open('wb') as f:
        pickle.dump(index, f, protocol=pickle.HIGHEST_PROTOCOL)
    print('Completed image scan:', scanned, 'metadata images')
else:
    print('Image matching disabled; image objects will remain unmatched.')


Resolving data files:   0%|          | 0/114 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/114 [00:00<?, ?it/s]

Images scanned: 1000 raw matches: 82 pixel matches: 0
Images scanned: 2000 raw matches: 179 pixel matches: 0
Images scanned: 3000 raw matches: 279 pixel matches: 0
Images scanned: 4000 raw matches: 366 pixel matches: 0
Images scanned: 5000 raw matches: 450 pixel matches: 0
Images scanned: 6000 raw matches: 558 pixel matches: 0
Images scanned: 7000 raw matches: 627 pixel matches: 0
Images scanned: 8000 raw matches: 730 pixel matches: 0
Images scanned: 9000 raw matches: 814 pixel matches: 0
Images scanned: 10000 raw matches: 877 pixel matches: 0
Images scanned: 11000 raw matches: 983 pixel matches: 0
Images scanned: 12000 raw matches: 1059 pixel matches: 0
Images scanned: 13000 raw matches: 1146 pixel matches: 0
Images scanned: 14000 raw matches: 1221 pixel matches: 0
Images scanned: 15000 raw matches: 1301 pixel matches: 0
Images scanned: 16000 raw matches: 1393 pixel matches: 0
Images scanned: 17000 raw matches: 1573 pixel matches: 0
Images scanned: 18000 raw matches: 1718 pixel matche

## Conservative caption retrieval and relationship inference

Retrieval priority is exact caption, relaxed-exact caption, then high-threshold fuzzy caption. Fuzzy candidates are restricted through uncommon tokens. Ambiguous candidate sets are accepted only when every candidate combination implies the same relationship.


In [7]:
TOKEN_RE = re.compile(r'[a-z0-9][a-z0-9+_.-]{3,}')
token_rows = defaultdict(list)
if ENABLE_FUZZY_CAPTIONS:
    for row_id, rec in enumerate(index['records']):
        for token in set(TOKEN_RE.findall(rec['relaxed_caption'])):
            token_rows[token].append(row_id)
    token_rows = {k:v for k,v in token_rows.items() if 2 <= len(v) <= MAX_FUZZY_CANDIDATES}
    print('Useful fuzzy-caption tokens:', len(token_rows))

retrieval_cache = {}
def retrieve_object(value):
    cache_key = hashlib.sha256(str(value).encode('utf-8')).hexdigest()
    if cache_key in retrieval_cache:
        return retrieval_cache[cache_key]
    if is_image(value):
        try:
            raw = decode_image_bytes(value)
            candidates = tuple(index['raw_images'].get(sha256_bytes(raw), ()))
            method = 'image_raw_exact' if len(candidates)==1 else 'image_raw_ambiguous' if candidates else 'image_unmatched'
            if not candidates and ENABLE_PIXEL_HASH:
                candidates = tuple(index['pixel_images'].get(pixel_hash(raw), ()))
                method = 'image_pixel_exact' if len(candidates)==1 else 'image_pixel_ambiguous' if candidates else 'image_unmatched'
        except Exception:
            method, candidates = 'invalid_image', ()
    else:
        exact = normalize_caption(value)
        candidates = tuple(index['caption_exact'].get(exact, ()))
        method = 'caption_exact' if len(candidates)==1 else 'caption_exact_ambiguous' if candidates else ''
        if not candidates:
            relaxed = relaxed_caption(value)
            candidates = tuple(index['caption_relaxed'].get(relaxed, ()))
            method = 'caption_relaxed_exact' if len(candidates)==1 else 'caption_relaxed_ambiguous' if candidates else ''
        if not candidates and ENABLE_FUZZY_CAPTIONS:
            query = relaxed_caption(value)
            tokens = sorted(set(TOKEN_RE.findall(query)), key=lambda t: len(token_rows.get(t, ())))
            pool = set()
            for token in tokens:
                pool.update(token_rows.get(token, ()))
                if len(pool) >= 50:
                    break
            scored = sorted(((fuzzy_ratio(query, index['records'][i]['relaxed_caption']), i) for i in pool), reverse=True)
            best = scored[0][0] if scored else 0.0
            second = scored[1][0] if len(scored)>1 else 0.0
            if best >= FUZZY_MIN_SCORE and best-second >= FUZZY_MIN_MARGIN:
                candidates, method = (scored[0][1],), 'caption_fuzzy_high_confidence'
            else:
                candidates, method = (), 'caption_unmatched'
        elif not candidates:
            method = 'caption_unmatched'
    result = {'method': method, 'candidates': candidates}
    retrieval_cache[cache_key] = result
    return result

def relation(left, right):
    if left['meta_row'] == right['meta_row']:
        return 'same_figure'
    if left['doi'] and left['doi'] == right['doi']:
        return 'same_paper'
    linked = (right['doi'] in left['references'] or right['doi'] in left['citations'] or
              left['doi'] in right['references'] or left['doi'] in right['citations'])
    return 'related_papers' if linked else 'unrelated_papers'

def infer_from_candidates(a, b):
    if not a['candidates'] or not b['candidates']:
        return 'unmatched', 'unresolved'
    outcomes = {relation(index['records'][i], index['records'][j]) for i in a['candidates'] for j in b['candidates']}
    if len(outcomes) != 1:
        return 'unmatched', 'ambiguous_disagreement'
    resolution = 'unique' if len(a['candidates'])==len(b['candidates'])==1 else 'candidate_consensus'
    return outcomes.pop(), resolution


Useful fuzzy-caption tokens: 33251


## Match train and test, then save compact audits


In [8]:
def record_fields(prefix, match):
    unique = match['candidates'][0] if len(match['candidates']) == 1 else None
    rec = index['records'][unique] if unique is not None else None
    return {
        f'{prefix}_method': match['method'], f'{prefix}_candidate_count': len(match['candidates']),
        f'{prefix}_meta_row': '' if rec is None else rec['meta_row'],
        f'{prefix}_uuid': '' if rec is None else rec['uuid'],
        f'{prefix}_image_id': '' if rec is None else rec['image_id'],
        f'{prefix}_doi': '' if rec is None else rec['doi'],
    }

def match_csv(path, output, labeled):
    if output.exists():
        output.unlink()
    usecols = ['id', 'obj_1', 'obj_2'] + (LABELS if labeled else [])
    first, processed = True, 0
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=CSV_CHUNK, keep_default_na=False):
        out = []
        for row in chunk.itertuples(index=False):
            a, b = retrieve_object(row.obj_1), retrieve_object(row.obj_2)
            inferred, resolution = infer_from_candidates(a, b)
            truth = LABELS[int(np.argmax([getattr(row, label) for label in LABELS]))] if labeled else ''
            item = {
                'id': row.id,
                'modality': 'image-image' if is_image(row.obj_1) and is_image(row.obj_2) else 'text-image' if is_image(row.obj_1)^is_image(row.obj_2) else 'text-text',
                'inferred_relationship': inferred, 'resolution': resolution,
                'true_relationship': truth,
                'metadata_correct': '' if not truth or inferred=='unmatched' else inferred==truth,
            }
            item.update(record_fields('obj_1', a)); item.update(record_fields('obj_2', b))
            out.append(item)
        pd.DataFrame(out).to_csv(output, mode='w' if first else 'a', header=first, index=False)
        first = False; processed += len(chunk)
        if processed % 1000 < len(chunk):
            print(output.name, 'rows:', processed)
    return pd.read_csv(output, keep_default_na=False)

train_matches = match_csv(TRAIN_CSV, WORK/'train_retrieval.csv', True)
test_matches = match_csv(TEST_CSV, WORK/'test_retrieval.csv', False)
print('Saved retrieval files.')


train_retrieval.csv rows: 1024
train_retrieval.csv rows: 2016
train_retrieval.csv rows: 3008
train_retrieval.csv rows: 4000
train_retrieval.csv rows: 5024
train_retrieval.csv rows: 6016
train_retrieval.csv rows: 7008
train_retrieval.csv rows: 8000
train_retrieval.csv rows: 9024
train_retrieval.csv rows: 10000
test_retrieval.csv rows: 1024
test_retrieval.csv rows: 2016
test_retrieval.csv rows: 3008
test_retrieval.csv rows: 4000
test_retrieval.csv rows: 5024
test_retrieval.csv rows: 6016
test_retrieval.csv rows: 7008
test_retrieval.csv rows: 8000
test_retrieval.csv rows: 9024
test_retrieval.csv rows: 10000
Saved retrieval files.


## Final report and decision gate

A retrieval method should be used as a hard override only when its labeled-train precision is extremely high. Test coverage, not the leaderboard, decides whether metadata retrieval is the main system or merely an auxiliary component.


In [9]:
def summarize_matches(frame, labeled):
    resolved = frame.inferred_relationship.isin(LABELS)
    result = {
        'rows': len(frame), 'resolved': int(resolved.sum()),
        'coverage': float(resolved.mean()),
        'resolution_counts': frame.resolution.value_counts().to_dict(),
        'relationship_counts': frame.inferred_relationship.value_counts().to_dict(),
        'coverage_by_modality': frame.assign(resolved=resolved).groupby('modality').resolved.mean().to_dict(),
        'obj_1_methods': frame.obj_1_method.value_counts().to_dict(),
        'obj_2_methods': frame.obj_2_method.value_counts().to_dict(),
    }
    if labeled and resolved.any():
        result['accuracy_when_resolved'] = float((frame.loc[resolved, 'inferred_relationship'] == frame.loc[resolved, 'true_relationship']).mean())
        result['accuracy_by_resolution'] = {
            name: float((part.inferred_relationship == part.true_relationship).mean())
            for name, part in frame.loc[resolved].groupby('resolution')
        }
    return result

summary = {
    'configuration': {
        'hf_dataset': HF_DATASET, 'hf_split': HF_SPLIT, 'metadata_path': str(METADATA_PATH),
        'pixel_hashes': ENABLE_PIXEL_HASH, 'fuzzy_captions': ENABLE_FUZZY_CAPTIONS,
        'fuzzy_min_score': FUZZY_MIN_SCORE, 'fuzzy_min_margin': FUZZY_MIN_MARGIN,
    },
    'data_audit': {
        'train_rows': train_audit['rows'], 'test_rows': test_audit['rows'],
        'train_classes': train_audit['class_counts'],
        'train_modalities': train_audit['modality_counts'], 'test_modalities': test_audit['modality_counts'],
    },
    'train_retrieval': summarize_matches(train_matches, True),
    'test_retrieval': summarize_matches(test_matches, False),
}
with (WORK/'audit_summary.json').open('w') as f:
    json.dump(summary, f, indent=2, sort_keys=True)
print(json.dumps(summary, indent=2, sort_keys=True))

train_precision = summary['train_retrieval'].get('accuracy_when_resolved', 0.0)
test_coverage = summary['test_retrieval']['coverage']
print('\nDECISION')
if train_precision < 0.995:
    print('STOP: retrieval precision is below 99.5%; inspect errors before using metadata overrides.')
elif test_coverage >= 0.90:
    print('Metadata retrieval is viable as the primary system. Build metadata-only and hybrid submissions next.')
elif test_coverage >= 0.20:
    print('Use metadata only as a high-confidence hybrid override; train a fallback for the remaining rows.')
else:
    print('Test metadata coverage is low. Preserve this result as an ablation and focus on the learned fallback.')
print('Artifacts:', sorted(str(p) for p in WORK.iterdir()))


{
  "configuration": {
    "fuzzy_captions": true,
    "fuzzy_min_margin": 2.0,
    "fuzzy_min_score": 98.5,
    "hf_dataset": "adsabs/AstroCLIMB",
    "hf_split": "train",
    "metadata_path": "None",
    "pixel_hashes": false
  },
  "data_audit": {
    "test_modalities": {
      "image-image": 3000,
      "text-image": 4000,
      "text-text": 3000
    },
    "test_rows": 10000,
    "train_classes": {
      "related_papers": 3000,
      "same_figure": 1000,
      "same_paper": 3000,
      "unrelated_papers": 3000
    },
    "train_modalities": {
      "image-image": 3000,
      "text-image": 4000,
      "text-text": 3000
    },
    "train_rows": 10000
  },
  "test_retrieval": {
    "coverage": 0.0,
    "coverage_by_modality": {
      "image-image": 0.0,
      "text-image": 0.0,
      "text-text": 0.0
    },
    "obj_1_methods": {
      "caption_exact": 3,
      "caption_unmatched": 6997,
      "image_unmatched": 3000
    },
    "obj_2_methods": {
      "caption_fuzzy_high_confidence"

## What to send back after the run

Download or attach these three small files:

- `audit_summary.json`
- `train_retrieval.csv`
- `test_retrieval.csv`

Also save the entire `astroclimb_01` output directory as a private Kaggle Dataset so the expensive metadata index can be reused. Do not publish it until the competition rules and organizers confirm that derived test artifacts may be shared.
